# Image Generation From Sentences

Ce notebook prend une liste de **phrases en français**, les traduit automatiquement en anglais (les modèles donnent de meilleurs résultats en anglais), puis génère pour chaque phrase une **image correspondante** via l'API publique et gratuite [Pollinations.ai](https://pollinations.ai/) (aucune clé API requise).

Les images sont sauvegardées dans le dossier `outputs/` et affichées directement dans le notebook.

## Étapes
1. Installation des dépendances
2. Chargement des phrases (depuis `phrases.txt` ou une liste Python)
3. Traduction FR → EN
4. Génération / téléchargement d'une image par phrase
5. Affichage des résultats

## 1. Installation des dépendances

In [ ]:
# À exécuter une seule fois
%pip install --quiet requests pillow deep-translator

In [ ]:
import os
import re
import time
import urllib.parse
from pathlib import Path

import requests
from PIL import Image
from IPython.display import display
from deep_translator import GoogleTranslator

# Dossier de sortie
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Dossier de sortie : {OUTPUT_DIR.resolve()}")

## 2. Chargement des phrases

Tu peux soit :
- Modifier le fichier `phrases.txt` (une phrase par ligne) — utilisé par défaut
- Soit définir directement une liste Python dans la cellule ci-dessous

In [ ]:
def load_phrases_from_file(path="phrases.txt"):
    """Charge les phrases depuis un fichier texte (une par ligne)."""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

phrases = load_phrases_from_file("phrases.txt")

# Alternative : remplacer par ta propre liste
# phrases = [
#     "un dragon qui crache du feu au-dessus d'un château",
#     "un robot qui jardine dans un champ de tournesols",
# ]

print(f"{len(phrases)} phrase(s) chargée(s) :")
for i, p in enumerate(phrases, 1):
    print(f"  {i}. {p}")

## 3. Traduction FR → EN

Les modèles de génération d'images comprennent mieux l'anglais. On utilise `deep-translator` (basé sur Google Translate, gratuit et sans clé).

In [ ]:
def translate_fr_to_en(text):
    """Traduit une phrase du français vers l'anglais. Retourne le texte original en cas d'erreur."""
    try:
        return GoogleTranslator(source="fr", target="en").translate(text)
    except Exception as e:
        print(f"  [Avertissement] Traduction échouée pour '{text}' : {e}")
        return text

translated = [translate_fr_to_en(p) for p in phrases]

for fr, en in zip(phrases, translated):
    print(f"FR : {fr}\nEN : {en}\n")

## 4. Génération des images via Pollinations.ai

L'URL `https://image.pollinations.ai/prompt/<prompt>` renvoie directement une image PNG générée par IA à partir du prompt.

Paramètres optionnels :
- `width`, `height` : dimensions
- `seed` : pour reproduire la même image
- `nologo=true` : pas de filigrane
- `model` : `flux` (par défaut, recommandé), `turbo`, etc.

In [ ]:
def slugify(text, max_len=60):
    """Convertit un texte en nom de fichier sûr."""
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE).strip().lower()
    text = re.sub(r"[\s-]+", "_", text)
    return text[:max_len] or "image"


def generate_image(
    prompt,
    output_path,
    width=768,
    height=768,
    model="flux",
    seed=None,
    timeout=120,
):
    """Génère une image via Pollinations.ai et la sauvegarde sur disque."""
    encoded_prompt = urllib.parse.quote(prompt)
    url = (
        f"https://image.pollinations.ai/prompt/{encoded_prompt}"
        f"?width={width}&height={height}&model={model}&nologo=true"
    )
    if seed is not None:
        url += f"&seed={seed}"

    response = requests.get(url, timeout=timeout)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path

In [ ]:
# Boucle principale : une image par phrase
results = []

for idx, (fr, en) in enumerate(zip(phrases, translated), start=1):
    filename = f"{idx:02d}_{slugify(fr)}.png"
    output_path = OUTPUT_DIR / filename

    print(f"[{idx}/{len(phrases)}] Génération : {fr}")
    print(f"    Prompt EN : {en}")

    try:
        generate_image(en, output_path)
        print(f"    Sauvegardée : {output_path}\n")
        results.append({"fr": fr, "en": en, "path": output_path, "ok": True})
    except Exception as e:
        print(f"    [Erreur] {e}\n")
        results.append({"fr": fr, "en": en, "path": None, "ok": False})

    # Petite pause pour éviter de saturer l'API
    time.sleep(1)

ok_count = sum(1 for r in results if r["ok"])
print(f"Terminé : {ok_count}/{len(results)} image(s) générée(s).")

## 5. Affichage des images générées

In [ ]:
for r in results:
    if not r["ok"]:
        continue
    print(f"Phrase : {r['fr']}")
    img = Image.open(r["path"])
    # Aperçu réduit pour le notebook
    img.thumbnail((512, 512))
    display(img)
    print(f"Fichier : {r['path']}\n" + "-" * 60)

## Bonus : générer une image pour une phrase unique

Cellule rapide pour tester une phrase à la volée.

In [ ]:
ma_phrase = "un château flottant dans les nuages au coucher du soleil"

en = translate_fr_to_en(ma_phrase)
print(f"Prompt EN : {en}")

out = OUTPUT_DIR / f"single_{slugify(ma_phrase)}.png"
generate_image(en, out)

img = Image.open(out)
img.thumbnail((512, 512))
display(img)
print(f"Sauvegardée : {out}")